### The first stage of processing. Here we download latest human core metadata from Metalog

In [ ]:
import requests
from gzip import GzipFile
import pandas as pd
from pathlib import Path
import os


def download_wide_latest() -> pd.DataFrame:
    with requests.get("https://metalog.embl.de//static/download/metadata/human_core_wide_latest.tsv.gz", stream=True) as req:
    # with requests.get("https://metalog.embl.de/api/samples/human.tsv", stream=True) as req:
        with GzipFile(fileobj=req.raw) as gzip:
            return pd.read_csv(gzip, delimiter="\t", low_memory=False) #type: ignore

# def download_genomes() -> pd.DataFrame:
#     with requests.get("https://black.embl.de/~fullam/spire/metadata/spire_v1_genome_metadata.tsv.gz", stream=True) as req:
#     # with requests.get("https://metalog.embl.de/api/samples/human.tsv", stream=True) as req:
#         with GzipFile(fileobj=req.raw) as gzip:
#             return pd.read_csv(gzip, delimiter="\t", low_memory=False) #type: ignore


inputs_path = Path("inputs")
intr_path = Path("intermediates")
output_path = Path("output")
figures_path = output_path / "figures"

os.makedirs(intr_path, exist_ok=True)
os.makedirs(figures_path, exist_ok=True)

core_wide = download_wide_latest()
# genomes = download_genomes()

### Scraping MAGs from SPIRE

Currently, SPIRE doesn't expose complete metadata of all the genomes they host. The file they provide contains weird data, and many genomes are missing there although certainly present when seeking manually. That's what we do here. This takes a while to complete, and you probably would need to change `concurrency` parameter to match your bandwidth and RAM capabilities (I used 128 with 32 GB). The scraped genomes are stored in the intermediate file `core_scraped_genomes.csv`
https://github.com/grp-bork/spire_contribute/issues/11

Will take a while to complete.

In [ ]:
from src.spire_scraper import scrape_multiproc

genomes_scraped_path = intr_path / "core_scraped_genomes.csv"

if not genomes_scraped_path.is_file():
    genomes = scrape_multiproc(core_wide, concurrency=128)
    genomes.to_csv(genomes_scraped_path)
else:
    genomes = pd.read_csv(genomes_scraped_path)

# core_wide.set_index("spire_sample_name").join(genomes.set_index("derived_from_sample"), how="inner")

In [ ]:
genomes

In [ ]:
min_completeness = 90
max_contamination = 5

random_state = 999

# Filtering the genomes by the target
hq_genomes = genomes[
    (genomes["completeness"] >= min_completeness) & 
    (genomes["contamination"] <= max_contamination)]\
    .sort_values(by="spire_id")
hq_genomes

In [ ]:
trait_columns = ["sex", "age_years", "age_category", "weight_kg", "bmi", "height_cm", "diet", "smoker", "subject_disease_status", "intervention", "type_of_birth", "medication_with_parents", "medication"]
sample_info_columns = ["doi", "sample_id", "sample_alias", "longitude", "latitude", "geographic_location", "subject_id", "spire_id"]

target_package = "human-gut"
target_material = "fecal material [ENVO:00002003]"
target_age = (18, 90)

# Filtering by the target age population and sample source. Including only unnecessary columns.
hq_fitting_genomes = hq_genomes[
    (hq_genomes["age_years"].isna() != True) &
    (hq_genomes["environmental_package"] == target_package) &
    (hq_genomes["environment_material"] == target_material) &
    (hq_genomes["age_years"].between(*target_age))]
hq_fitting_genomes = hq_fitting_genomes[trait_columns + sample_info_columns]
hq_fitting_genomes

In [ ]:
# Grouping by sample_alias to obtain samples the genomes are coming from.
fitting_samples = hq_fitting_genomes.groupby("sample_alias").first()
fitting_samples

In [ ]:
# Deduplicating samples which come from the same subject.
deduplicated_samples = fitting_samples\
    .groupby("subject_id")\
    .sample(1, random_state=random_state)
deduplicated_samples

In [ ]:
hq_genomes_dedup_samples = hq_fitting_genomes[hq_fitting_genomes["sample_alias"].isin(deduplicated_samples.index)]
hq_genomes_dedup_samples

In [ ]:
def balance_sex(group):
    counts = group["sex"].value_counts()
    nice_count = counts.min() if len(counts) == 2 else 0
    return group.groupby("sex").sample(n=nice_count, random_state=random_state)

# balancing by sex.
balanced_samples_df: pd.DataFrame = deduplicated_samples\
    .groupby("geographic_location")\
    .apply(balance_sex)\
    .reset_index(level=0) # type: ignore
balanced_samples_df

### Supplementary Table 1

In [ ]:
grouping = deduplicated_samples\
    .groupby("geographic_location")\
    .size()

participants_column = "Participants"

all_goods = hq_genomes_dedup_samples.groupby("geographic_location")
supplementary_table_1 = pd.DataFrame(grouping, columns=[participants_column])

males_column = "Males"
females_column = "Females"
mags_column = "Mags"
mags_per_sample_column = "Mags per sample"

supplementary_table_1[[males_column, females_column]] = deduplicated_samples\
    .groupby("geographic_location")["sex"]\
    .value_counts()\
    .unstack(fill_value=0)[["male", "female"]]
supplementary_table_1[mags_column] = all_goods.size()
supplementary_table_1[mags_per_sample_column] = hq_genomes_dedup_samples\
    .groupby(["geographic_location", "sample_alias"])\
    .size()\
    .groupby("geographic_location")\
    .apply(lambda x: f"{x.mean():.2f} ± {x.std():.2f}")
supplementary_table_1.index = supplementary_table_1.index.rename("Country")

supplementary_table_1.sort_values(by=participants_column, ascending=False).head(15)

### Table 2

In [ ]:
country_grouping = balanced_samples_df.groupby("geographic_location")

table_2 = pd.DataFrame(
    balanced_samples_df\
        .reset_index()\
        .groupby(["geographic_location", "sample_alias"])\
        .size()\
        .unstack(0)\
        .sum(), columns=["Samples count"])
table_2["Participant age"] = country_grouping["age_years"].apply(lambda x: f"{x.mean():.2f} ± {x.std():.2f}")
table_2["Male%"] = (country_grouping["sex"].value_counts().unstack()["male"] / country_grouping.size())\
    .apply(lambda x: f"{x * 100:.2f}")
table_2.index = table_2.index.rename("Country")
table_2.sort_values("Samples count", ascending=False)

In [ ]:
hq_balanced_genomes = hq_genomes_dedup_samples[hq_genomes_dedup_samples["sample_alias"].isin(balanced_samples_df.index)]
hq_balanced_genomes

In [ ]:
picked_countries = ["China", "USA", "Netherlands"]

hq_picked_genomes = hq_balanced_genomes[hq_balanced_genomes["geographic_location"].isin(picked_countries)]
hq_picked_genomes

### Population structure panel

In [ ]:
from src.figures import create_pyramid_panel

create_pyramid_panel(
    figures_path / "pyramid_panel.svg", 
    balanced_samples_df, 
    picked_countries)

In [ ]:
mags_to_classify_path = intr_path / "mags_to_classify.csv"

hq_picked_genomes.to_csv(mags_to_classify_path, index=False)

### Downloading the MAGs themselves from SPIRE. 
Make sure to tune `concurrency` here too. It should probably be similar to the one used for scraping the metadata. Will take a while to complete.

In [ ]:
from src.spire_downloader import fetch_mags


mag_files = intr_path / "mags"

mags_to_download = hq_picked_genomes["spire_id"]

mags_to_download = mags_to_download[mags_to_download.apply(lambda x: not (mag_files / f"{x}.fa").is_file())]

fetch_mags(mags_to_download, mag_files, 128)